#**Ariel Pan, Daniel Lyu**

Spring 2026

CS 443: Bio-inspired Machine Learning

Project 2: Predictive Coding

**#### Extension: Explore more of ConvPCN**

Inspired by the paper **Deep Predictive Coding Network with Local Recurrent Processing for Object Recognition**

Main Changes:
- add an explicit **bypass / skip path** inside each recurrent block
- vary the **pooling schedule**
- vary the **depth / width**

**bypass layer:**

one path: recurrent predictive coding update

second path: direct bypass of lower-level features

final output: combine the two

## Compare multiple architecture variants

**num_steps**: The number of recurrent predictive-coding update iterations performed inside each block during one forward pass; larger values allow more error refinement but increase runtime.


**PCN Units**: The number of convolutional channels in each PCN block, which controls the representational width and capacity of the network at different depths.


**MaxPool After Block**: A tuple indicating whether max pooling is applied after each block, which reduces spatial resolution and increases the effective receptive field in deeper layers.


**Bypass (Skip)**: Whether each block includes a shortcut path that carries lower-level features directly forward, improving information flow and making optimization easier.


**Depth (#blocks)**: The total number of predictive-coding blocks stacked in the architecture, which determines how hierarchical and deep the model is.

## 1) Imports and setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)


plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.show()
plt.rcParams.update({'font.size': 18})

np.set_printoptions(suppress=True, precision=3)

%load_ext autoreload
%autoreload 2

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
project_path = '/content/drive/MyDrive/ColabNotebooks/project2'
os.chdir(project_path)

In [4]:
from conv_pcn_ext7 import (
    ConvPCNExt7,
    build_ext7_small,
    build_ext7_medium,
    build_ext7_deeper,
)
from conv_pcn_ext7_helpers import (
    set_global_seed,
    make_cifar_subset,
    describe_split,
    default_experiment_configs,
    run_with_project_api,
)

set_global_seed(42)
print(tf.__version__)

2.18.0


## 2) Load CIFAR dev data

In [5]:
from image_datasets import get_dataset, train_val_split
x_train, y_train, x_test, y_test = get_dataset('cifar10',flatten=False)


x_train shape: (50000, 32, 32, 3), dtype: <dtype: 'float32'>
y_train shape: (50000,), dtype: <dtype: 'int32'>
x_test shape: (10000, 32, 32, 3), dtype: <dtype: 'float32'>
y_test shape: (10000,), dtype: <dtype: 'int32'>


In [6]:
N = 3000

# random subset indices
idx = np.random.choice(len(x_train), size=N, replace=False)

x_sub = tf.gather(x_train, idx)
y_sub = tf.gather(y_train, idx)

val_fraction = 0.2
val_size = int(N * val_fraction)

x_val = tf.constant(x_sub[:val_size], dtype=tf.float32)
y_val = tf.constant(y_sub[:val_size], dtype=tf.int32)

x_train_small = tf.constant(x_sub[val_size:], dtype=tf.float32)
y_train_small = tf.constant(y_sub[val_size:], dtype=tf.int32)

print("train:", x_train_small.shape)
print("val:", x_val.shape)

class Split:
    def __init__(self, x_train, y_train, x_val, y_val):
        self.x_train = x_train
        self.y_train = y_train
        self.x_val = x_val
        self.y_val = y_val

split = Split(x_train_small, y_train_small, x_val, y_val)

train: (2400, 32, 32, 3)
val: (600, 32, 32, 3)


## 4) Sanity-check one model instance


In [7]:
num_classes = int(tf.reduce_max(y_train_small).numpy()) + 1
input_shape = tuple(x_train_small.shape[1:])

model = build_ext7_small(input_shape, num_classes)
print(model)

# Forward-pass sanity check on a tiny batch
sanity_logits = model(split.x_train[:4])
print('sanity output shape:', sanity_logits.shape)

sanity output shape: (4, 10)


## 5) What the modified block is doing

1. **Feedforward conv** to initialize the latent state
2. **Feedback transposed conv** to reconstruct the lower layer
3. **1x1 bypass conv** to carry the lower-layer signal directly to the block output


In [8]:
from conv_pcn_ext7_blocks import ConvPCNBlockBypass
#help(ConvPCNBlockBypass)

## 6) Train a single fast baseline-style Extension 7 model


In [9]:
print("x_train:", split.x_train.shape, "y_train:", split.y_train.shape)
print("x_val:", split.x_val.shape, "y_val:", split.y_val.shape)

x_train: (2400, 32, 32, 3) y_train: (2400,)
x_val: (600, 32, 32, 3) y_val: (600,)


In [10]:
small_model = build_ext7_small(input_shape, num_classes)

small_history, small_result = run_with_project_api(
    model=small_model,
    split=split,
    epochs=5,
    batch_size=32,
    lr=1e-3,
)

print(small_result)

---------------------------------------------------------------------------
Dense layer output(Dense_output) shape: [1, 10]
Dense layer output(Dense_hidden) shape: [1, 96]
Dropout layer output(Dropout) shape: [1, 8192]
Flatten layer output(Flatten) shape: [1, 8192]
MaxPool2D layer output(MaxPool_2) shape: [1, 8, 8, 128]
PCNBlockExt7_2:
	Conv2D layer output(PCNBlockExt7_2_bypass) shape: [1, 16, 16, 128]
	Conv2DTranspose layer output(PCNBlockExt7_2_convT) shape: [1, 16, 16, 96]
	Conv2D layer output(PCNBlockExt7_2_conv) shape: [1, 16, 16, 128]
MaxPool2D layer output(MaxPool_1) shape: [1, 16, 16, 96]
PCNBlockExt7_1:
	Conv2D layer output(PCNBlockExt7_1_bypass) shape: [1, 32, 32, 96]
	Conv2DTranspose layer output(PCNBlockExt7_1_convT) shape: [1, 32, 32, 64]
	Conv2D layer output(PCNBlockExt7_1_conv) shape: [1, 32, 32, 96]
PCNBlockExt7_0:
	Conv2D layer output(PCNBlockExt7_0_bypass) shape: [1, 32, 32, 64]
	Conv2DTranspose layer output(PCNBlockExt7_0_convT) shape: [1, 32, 32, 32]
	Conv2D layer o

In [17]:
import pandas as pd

data = [
    {
        "Model": "Vanilla (Original)",
        "num_steps": 5,
        "PCN Units": "(64, 128, 192)",
        "MaxPool After Block": "(True, True, True)",
        "Bypass (Skip)": "No",
        "Depth (#blocks)": 3,
    },
    {
        "Model": "Ext7 Small",
        "num_steps": 1,
        "PCN Units": "(64, 96, 128)",
        "MaxPool After Block": "(False, True, True)",
        "Bypass (Skip)": "Yes",
        "Depth (#blocks)": 3,
    },
    {
        "Model": "Ext7 Medium",
        "num_steps": 5,
        "PCN Units": "(64, 128, 128, 192)",
        "MaxPool After Block": "(False, False, True, True)",
        "Bypass (Skip)": "Yes",
        "Depth (#blocks)": 4,
    },
    {
        "Model": "Ext7 Deeper",
        "num_steps": 7,
        "PCN Units": "(64, 96, 128, 192, 192)",
        "MaxPool After Block": "(False, True, False, True, True)",
        "Bypass (Skip)": "Yes",
        "Depth (#blocks)": 5,
    },
]

df = pd.DataFrame(data)
df


,Model,num_steps,PCN Units,MaxPool After Block,Bypass (Skip),Depth (#blocks)
0,Vanilla (Original),5,"(64, 128, 192)","(True, True, True)",No,3
1,Ext7 Small,1,"(64, 96, 128)","(False, True, True)",Yes,3
2,Ext7 Medium,5,"(64, 128, 128, 192)","(False, False, True, True)",Yes,4
3,Ext7 Deeper,7,"(64, 96, 128, 192, 192)","(False, True, False, True, True)",Yes,5


In [11]:
results = []
configs = default_experiment_configs()

builders = {
    'build_ext7_small': build_ext7_small,
    'build_ext7_medium': build_ext7_medium,
    'build_ext7_deeper': build_ext7_deeper,
}

for cfg in configs:
    print('Running config:', cfg['name'])

    model = builders[cfg['builder']](input_shape, num_classes)
    _, result = run_with_project_api(
        model=model,
        split=split,
        epochs=cfg['epochs'],
        batch_size=32,
        lr=cfg['lr'],
    )
    result.name = cfg['name']
    results.append(result)
    print(result)

Running config: small_fast
---------------------------------------------------------------------------
Dense layer output(Dense_output) shape: [1, 10]
Dense layer output(Dense_hidden) shape: [1, 96]
Dropout layer output(Dropout) shape: [1, 8192]
Flatten layer output(Flatten) shape: [1, 8192]
MaxPool2D layer output(MaxPool_2) shape: [1, 8, 8, 128]
PCNBlockExt7_2:
	Conv2D layer output(PCNBlockExt7_2_bypass) shape: [1, 16, 16, 128]
	Conv2DTranspose layer output(PCNBlockExt7_2_convT) shape: [1, 16, 16, 96]
	Conv2D layer output(PCNBlockExt7_2_conv) shape: [1, 16, 16, 128]
MaxPool2D layer output(MaxPool_1) shape: [1, 16, 16, 96]
PCNBlockExt7_1:
	Conv2D layer output(PCNBlockExt7_1_bypass) shape: [1, 32, 32, 96]
	Conv2DTranspose layer output(PCNBlockExt7_1_convT) shape: [1, 32, 32, 64]
	Conv2D layer output(PCNBlockExt7_1_conv) shape: [1, 32, 32, 96]
PCNBlockExt7_0:
	Conv2D layer output(PCNBlockExt7_0_bypass) shape: [1, 32, 32, 64]
	Conv2DTranspose layer output(PCNBlockExt7_0_convT) shape: [1, 

## 8) Turn results into a table

This cell makes it easier to read off your final report claims.
In particular, look for whether the bypass + pooling schedule helps validation accuracy enough to justify the extra runtime.

In [12]:
import pandas as pd

rows = []
for r in results:
    rows.append({
        'name': r.name,
        'runtime_sec': r.runtime_sec,
        'epoch_used': r.epoch_used,
        'final_val_acc': r.final_val_acc,
        'final_train_loss': r.final_train_loss,
        'final_val_loss': r.final_val_loss,
    })

results_df = pd.DataFrame(rows)
results_df.sort_values('final_val_acc', ascending=False)

,name,runtime_sec,epoch_used,final_val_acc,final_train_loss,final_val_loss
2,deeper_pooled,72.506282,13,0.555556,0.059754,1.849539
0,small_fast,14.881907,13,0.503472,0.047045,2.195986
1,medium_paper,43.460813,8,0.097222,32.804162,33.259562


In [13]:
from conv_pcn import ConvPCN7XL
big_model = ConvPCN7XL(
    input_feats_shape=split.x_train.shape[1:],
    C=10
)
big_model.compile(lr=1e-4)
btrain_loss, bval_loss, bval_acc, _ = big_model.fit(
    split.x_train, split.y_train,
    split.x_val, split.y_val,
    batch_size=32,
    max_epochs=50,
    patience=7,
    lr_patience=3,
    lr_max_decays=3,
)

---------------------------------------------------------------------------
Dense layer output(Dense_output) shape: [1, 10]
Dense layer output(Dense_hidden) shape: [1, 128]
Dropout layer output(Dropout) shape: [1, 16384]
Flatten layer output(Flatten) shape: [1, 16384]
MaxPool2D layer output(MaxPool_3) shape: [1, 8, 8, 256]
PCNBlock_3:
	Conv2DTranspose layer output(PCNBlock_3_convT) shape: [1, 16, 16, 256]
	Conv2D layer output(PCNBlock_3_conv) shape: [1, 16, 16, 256]
MaxPool2D layer output(MaxPool_2) shape: [1, 16, 16, 256]
PCNBlock_2:
	Conv2DTranspose layer output(PCNBlock_2_convT) shape: [1, 32, 32, 128]
	Conv2D layer output(PCNBlock_2_conv) shape: [1, 32, 32, 256]
PCNBlock_1:
	Conv2DTranspose layer output(PCNBlock_1_convT) shape: [1, 32, 32, 128]
	Conv2D layer output(PCNBlock_1_conv) shape: [1, 32, 32, 128]
PCNBlock_0:
	Conv2DTranspose layer output(PCNBlock_0_convT) shape: [1, 32, 32, 64]
	Conv2D layer output(PCNBlock_0_conv) shape: [1, 32, 32, 128]
Conv2D layer output(Conv2D_init) s